<a href="https://colab.research.google.com/github/owlmt/in_quest_of_entropy/blob/main/LinuxDRNG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
linux_rng_pipeline.py is a pure-Python, Colab-ready, high-fidelity educational
simulator of selected mechanisms in the modern Linux RNG (drivers/char/random.c,
post-5.18). It uses hardcoded event inputs so every stage is reproducible. It
implements selected algorithms and control-flow from the kernel: early-init
mixing, per-CPU fast-pool mixing, timer entropy estimation, BLAKE2s input-pool
conditioning, HKDF-like extraction, ChaCha20 fast key erasure, readiness states,
pre-ready output semantics, trust flags, and privileged trust boundaries.

It is NOT a bit-exact reproduction of any live kernel and must not be used to
claim prediction of real Linux RNG output. See REMAINING NON-BIT-EXACT AREAS.

SCOPE / GUARDRAILS
  * Pure stdlib (struct, json, hashlib for cross-checks only). Python 3.9+.
  * Does NOT read /dev/random, /dev/urandom, or call getrandom(). All "syscalls"
    are conceptual state transitions over hardcoded data.
  * All inputs are hardcoded: events, command line, UTS, arch-RNG/RDSEED
    stand-ins, and timing values.
  * Adversary-view ("honesty") accounting ILLUSTRATES the data-processing
    ceiling; it is not a claim about Linux's internal truth.
"""

import struct
import json
import hashlib   # used ONLY to cross-check our BLAKE2s in self-tests

# ===========================================================================
# 0.  VERSION PIN + TRUST FLAGS + GLOBAL CONFIG
# ===========================================================================
# Pin validated_against to an exact commit hash or distro source before
# treating ANY output as version-faithful. Until then a runtime warning fires.
TARGET_KERNEL = {
    "family": "Linux modern random.c post-5.18",
    "reference": "drivers/char/random.c",
    "validated_against": None,
    "required_action": "Set to exact commit hash or distro source before claiming version fidelity",
}

# Kernel boot trust parameters (random.trust_cpu / random.trust_bootloader) + hwrng.
TRUST_CPU = False          # random.trust_cpu : credit arch/CPU RNG at init?
TRUST_BOOTLOADER = False   # random.trust_bootloader : credit bootloader seed?
TRUST_HWRNG = True         # credit hw_random feeder (e.g. virtio-rng)?
CAP_SYS_ADMIN = True       # privileged ioctls (RNDADDTOENTCNT/RNDADDENTROPY) allowed?

# Behavior toggles. The two below default to MAINLINE-FAITHFUL behavior; the
# alternative (spec-requested) interpretations are available but are NOT what
# current drivers/char/random.c does -- see comments at the use sites.
MIX_COUNT_INTO_POOL = False     # mainline mixes only the 2 longs; True = older-writeup style
HARDIRQ_COUNT_INFLATION = False # mainline credits init_bits directly even in hardirq (see add_timer_randomness_model)
CLAMP_IRQ_CREDIT = True         # apply the FAST_POOL_WORDS_MIXED*64 clamp (modeling choice; mainline has no such clamp)

NCPU = 2
HZ = 1000
FLUSH_COUNT = 1024
FAST_POOL_WORDS_MIXED = 2       # mix_interrupt_randomness copies 2 of 4 longs
POOL_EARLY_BITS = 128
POOL_READY_BITS = 256
NUM_EARLY = 3                   # number of early-init pseudo-events (idx 0..2)

MASK64 = (1 << 64) - 1
MASK32 = (1 << 32) - 1

# Hardcoded early-init material (NOT real system values).
KERNEL_COMMAND_LINE = b"root=/dev/xvda1 console=ttyS0 random.trust_cpu=off random.trust_bootloader=off"
UTS_NAME = b"ip-172-31-4-240-amzn2023"
ARCH_EARLY_RANDOM = bytes.fromhex("deadbeefcafef00d0123456789abcdef")   # 16B arch RNG stand-in

# Per-event timing jitter (low cycle bits) -- the ONLY entropy-bearing part on a
# real machine. cycles(jpos) = CYC_BASE + jpos*CYC_STRIDE + JITTER[jpos].
CYC_BASE   = 0x00000003A1B40000
CYC_STRIDE = 0x000000000000C350
JITTER = [0x00d, 0x09e, 0x2a7, 0x071, 0x1c4, 0x033, 0x118, 0x205, 0x06f, 0x1aa,
          0x02e, 0x0f7, 0x261, 0x088, 0x1d3, 0x04c, 0x122, 0x2bb, 0x095, 0x1e0,
          0x057, 0x10c, 0x244, 0x07e, 0x199, 0x03b, 0x12d, 0x2a0, 0x066, 0x1bb,
          0x140, 0x289, 0x05a, 0x1ce, 0x0b3, 0x222, 0x07c, 0x191, 0x2d8, 0x048]

# arch RNG block consumed by extract_entropy (RDSEED/RDRAND/TSC fallback stand-in).
ARCH_RNG_BLOCK = bytes.fromhex("a1a1a1a1a1a1a1a1b2b2b2b2b2b2b2b2"
                               "c3c3c3c3c3c3c3c3d4d4d4d4d4d4d4d4")

def cycles_of(jpos, jitter):
    return (CYC_BASE + jpos * CYC_STRIDE + jitter[jpos]) & MASK64


# ===========================================================================
# 1.  CRYPTO CORES  (validated against KATs in self_tests)
# ===========================================================================
SIPHASH_CONST = [0x736f6d6570736575, 0x646f72616e646f6d,
                 0x6c7967656e657261, 0x7465646279746573]

def rotl64(x, b): return ((x << b) | (x >> (64 - b))) & MASK64
def swab64(x):    return int.from_bytes(x.to_bytes(8, 'big'), 'little')

def sipround(v):
    v[0] = (v[0] + v[1]) & MASK64; v[1] = rotl64(v[1], 13); v[1] ^= v[0]; v[0] = rotl64(v[0], 32)
    v[2] = (v[2] + v[3]) & MASK64; v[3] = rotl64(v[3], 16); v[3] ^= v[2]
    v[0] = (v[0] + v[3]) & MASK64; v[3] = rotl64(v[3], 21); v[3] ^= v[0]
    v[2] = (v[2] + v[1]) & MASK64; v[1] = rotl64(v[1], 17); v[1] ^= v[2]; v[2] = rotl64(v[2], 32)

def fast_mix(pool, m0, m1):
    pool[3] ^= m0; sipround(pool); pool[0] ^= m0
    pool[3] ^= m1; sipround(pool); pool[0] ^= m1

B2S_IV = [0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
          0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19]
B2S_SIGMA = [
 [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],[14,10,4,8,9,15,13,6,1,12,0,2,11,7,5,3],
 [11,8,12,0,5,2,15,13,10,14,3,6,7,1,9,4],[7,9,3,1,13,12,11,14,2,6,5,10,4,0,15,8],
 [9,0,5,7,2,4,10,15,14,1,11,12,6,8,3,13],[2,12,6,10,0,11,8,3,4,13,7,5,15,14,1,9],
 [12,5,1,15,14,13,4,10,0,7,6,3,9,2,8,11],[13,11,7,14,12,1,3,9,5,0,15,4,8,6,2,10],
 [6,15,14,9,11,3,0,8,12,2,13,7,1,4,10,5],[10,2,8,4,7,6,1,5,15,11,9,14,3,12,13,0]]

def rotr32(x, n): return ((x >> n) | (x << (32 - n))) & MASK32

class Blake2s:
    """BLAKE2s-256, RFC 7693, with keyed init and a clean snapshot()."""
    def __init__(self, key=b''):
        self.keylen = len(key)
        self.h = B2S_IV[:]
        self.h[0] ^= 0x01010000 ^ (self.keylen << 8) ^ 32
        self.t = 0
        self.buf = bytearray()
        if key:
            block = bytearray(64); block[:self.keylen] = key
            self.update(bytes(block))

    def _compress(self, blk, last):
        m = list(struct.unpack('<16I', blk))
        v = self.h + B2S_IV[:]
        v[12] ^= self.t & MASK32
        v[13] ^= (self.t >> 32) & MASK32
        if last: v[14] ^= MASK32
        def G(a, b, c, d, x, y):
            v[a] = (v[a] + v[b] + x) & MASK32; v[d] = rotr32(v[d] ^ v[a], 16)
            v[c] = (v[c] + v[d]) & MASK32;     v[b] = rotr32(v[b] ^ v[c], 12)
            v[a] = (v[a] + v[b] + y) & MASK32; v[d] = rotr32(v[d] ^ v[a], 8)
            v[c] = (v[c] + v[d]) & MASK32;     v[b] = rotr32(v[b] ^ v[c], 7)
        for r in range(10):
            s = B2S_SIGMA[r]
            G(0,4,8,12,  m[s[0]],  m[s[1]]);  G(1,5,9,13,  m[s[2]],  m[s[3]])
            G(2,6,10,14, m[s[4]],  m[s[5]]);  G(3,7,11,15, m[s[6]],  m[s[7]])
            G(0,5,10,15, m[s[8]],  m[s[9]]);  G(1,6,11,12, m[s[10]], m[s[11]])
            G(2,7,8,13,  m[s[12]], m[s[13]]); G(3,4,9,14,  m[s[14]], m[s[15]])
        for i in range(8): self.h[i] ^= v[i] ^ v[8+i]

    def update(self, data):
        for byte in data:
            if len(self.buf) == 64:
                self.t += 64; self._compress(bytes(self.buf), False); self.buf = bytearray()
            self.buf.append(byte)

    def snapshot(self):
        c = Blake2s.__new__(Blake2s)
        c.keylen = self.keylen; c.h = self.h[:]; c.t = self.t; c.buf = bytearray(self.buf)
        return c

    def final(self):
        self.t += len(self.buf)
        self.buf += b'\x00' * (64 - len(self.buf))
        self._compress(bytes(self.buf), True)
        return b''.join(struct.pack('<I', x) for x in self.h)

def blake2s(msg, key=b''):
    s = Blake2s(key); s.update(msg); return s.final()

CHACHA_CONST = [0x61707865, 0x3320646e, 0x79622d32, 0x6b206574]   # "expand 32-byte k"

def rotl32(x, n): return ((x << n) | (x >> (32 - n))) & MASK32

def chacha20_block(state):
    x = state[:]
    def QR(a, b, c, d):
        x[a] = (x[a] + x[b]) & MASK32; x[d] = rotl32(x[d] ^ x[a], 16)
        x[c] = (x[c] + x[d]) & MASK32; x[b] = rotl32(x[b] ^ x[c], 12)
        x[a] = (x[a] + x[b]) & MASK32; x[d] = rotl32(x[d] ^ x[a], 8)
        x[c] = (x[c] + x[d]) & MASK32; x[b] = rotl32(x[b] ^ x[c], 7)
    for _ in range(10):
        QR(0,4,8,12); QR(1,5,9,13); QR(2,6,10,14); QR(3,7,11,15)
        QR(0,5,10,15); QR(1,6,11,12); QR(2,7,8,13); QR(3,4,9,14)
    out = b''.join(struct.pack('<I', (x[i] + state[i]) & MASK32) for i in range(16))
    state[12] = (state[12] + 1) & MASK32
    return out

def chacha_init_state(key32):
    return CHACHA_CONST[:] + list(struct.unpack('<8I', key32)) + [0, 0, 0, 0]

def crng_fast_key_erasure(key32, out_len):
    state = chacha_init_state(key32)
    first_block = chacha20_block(state)            # state[12] -> 1
    return first_block[:32], state, first_block[32:32 + out_len]


# ===========================================================================
# 2.  HELPERS
# ===========================================================================
def hpool(p): return " ".join("%016x" % w for w in p)
def fls(x): return x.bit_length()
def hamming(a, b): return sum(bin(x ^ y).count("1") for x, y in zip(a, b))
def rule(c="="): return c * 114

def disk_event_num(devt):
    """Simplified stand-in for the disk timer 'num' (BSI uses 0x100 + disk_devt).
    NOT pinned to the exact major/minor encoding of any kernel; see fidelity warning."""
    return (0x100 + devt) & MASK32


# ===========================================================================
# 3.  STRUCTURED EVENT + ADVERSARY ACCOUNTING
# ===========================================================================
SOURCES = ("CMDLINE", "UTS", "ARCH_EARLY",
           "IRQ", "INPUT", "DISK", "DEVICE", "BOOTLOADER", "HWRNG",
           "VMFORK", "USER_WRITE", "ADMIN_CREDIT", "ADMIN_ADD_ENTROPY",
           "RNDRESEEDCRNG")

class Event:
    def __init__(self, idx, source, cpu=0, jiffies=None, payload=None, jpos=None,
                 credited_bits_claimed=None, assumed_hinf_given_VA=0,
                 field_class="public", adversary_known=True, context="process",
                 comment=""):
        assert source in SOURCES, "unknown source %s" % source
        self.idx = idx; self.source = source; self.cpu = cpu; self.jiffies = jiffies
        self.payload = payload or {}; self.jpos = jpos
        self.credited_bits_claimed = credited_bits_claimed
        self.assumed_hinf_given_VA = assumed_hinf_given_VA
        self.field_class = field_class            # public | predictable | opaque
        self.adversary_known = adversary_known
        self.context = context                     # hardirq | process
        self.comment = comment
        self.cycles = None; self.encoded_bits = 0; self.kernel_credit = 0
        self.ignored_repeat = False

    def to_dict(self):
        return {"idx": self.idx, "source": self.source, "cpu": self.cpu,
                "cycles": self.cycles, "jiffies": self.jiffies, "payload": self.payload,
                "context": self.context, "encoded_bits": self.encoded_bits,
                "kernel_credit": self.kernel_credit,
                "credited_bits_claimed": self.credited_bits_claimed,
                "assumed_Hinf_given_VA": self.assumed_hinf_given_VA,
                "field_class": self.field_class, "adversary_known": self.adversary_known,
                "ignored_repeat": self.ignored_repeat, "comment": self.comment}

class EntropyAccounting:
    """Illustrative adversary-view ledger. NOT a claim about Linux internals."""
    def __init__(self): self.rows = []
    def add(self, ev, notes=""):
        self.rows.append((ev.idx, ev.source, ev.encoded_bits, ev.kernel_credit,
                          ev.assumed_hinf_given_VA, ev.field_class, notes or ev.comment))
    @property
    def cum_encoded(self): return sum(r[2] for r in self.rows)
    @property
    def cum_credit(self):  return sum(r[3] for r in self.rows)
    @property
    def cum_hinf(self):    return sum(r[4] for r in self.rows)
    @property
    def honesty_gap(self): return self.cum_credit - self.cum_hinf


# ===========================================================================
# 4.  STATEFUL COMPONENTS
# ===========================================================================
class FastPool:
    def __init__(self, cpu):
        self.cpu = cpu
        self.pool = SIPHASH_CONST[:]
        self.count = 0
        self.last_flush_jiffies = 0
        self.mix_inflight = False
    def mix(self, m0, m1):
        fast_mix(self.pool, m0, m1); self.count += 1
    def read_two_longs(self):
        return (self.pool[0], self.pool[1])

class InputPool:
    """Continuous BLAKE2s-256 conditioning pool. Mixing != crediting."""
    def __init__(self):
        self.hash = Blake2s()
        self.log = []
    def _mix_pool_bytes(self, data):
        self.hash.update(data)
    def mix_pool_bytes(self, data, source, credited):
        before = self.hash.snapshot().final()[:8]
        self._mix_pool_bytes(data)
        after = self.hash.snapshot().final()[:8]
        self.log.append((source, len(data), before.hex(), after.hex(), credited))
    def seed(self):
        return self.hash.snapshot().final()
    def rekey(self, next_key):
        self.hash = Blake2s(key=next_key)
    def preview(self):
        return self.hash.snapshot().final()[:8].hex()

class BaseCRNG:
    def __init__(self):
        self.key = b'\x00' * 32; self.generation = 0
        self.birth_time = None; self.ready_state = "CRNG_EMPTY"

class PerCpuCRNG:
    def __init__(self, cpu):
        self.cpu = cpu; self.key = b'\x00' * 32; self.generation = -1; self.position = 0


# ===========================================================================
# 5.  THE EDUCATIONAL MODEL
# ===========================================================================
class LinuxRNGEducationalModel:
    def __init__(self, jitter, verbose=True):
        self.jitter = jitter; self.verbose = verbose
        self.fastpools = [FastPool(c) for c in range(NCPU)]
        self.input_pool = InputPool()
        self.acct = EntropyAccounting()
        self.init_bits = 0
        self.crng_state = "CRNG_EMPTY"
        self.base = BaseCRNG()
        self.percpu = [PerCpuCRNG(c) for c in range(NCPU)]
        self.timer_states = {}
        self.transitions = []
        self.reseed_seed = None; self.reseed_idx = None
        self.fastpool_rows = []; self.flush_rows = []; self.timer_rows = []
        self.early_init_rows = []; self.hardirq_rows = []; self.pre_ready_events = []
        self.denied_ioctls = []
        self.input_last_value = None
        self.vmfork_seen_pre_ready = False
        self._disk_warned = False

    def log(self, *a):
        if self.verbose: print(*a)

    # ---- crediting (separate from mixing) --------------------------------
    def credit_init_bits_model(self, bits, idx, action="credit"):
        before = self.init_bits; before_state = self.crng_state
        self.init_bits = min(POOL_READY_BITS, self.init_bits + bits)
        if before < POOL_EARLY_BITS <= self.init_bits and self.crng_state == "CRNG_EMPTY":
            self.crng_state = "CRNG_EARLY"; self.base.ready_state = "CRNG_EARLY"
        if before < POOL_READY_BITS <= self.init_bits and self.crng_state != "CRNG_READY":
            self.crng_state = "CRNG_READY"
            self.reseed_seed = self.input_pool.seed(); self.reseed_idx = idx
            self.crng_reseed_model(trigger="init_bits reached 256")
        self.transitions.append((idx, before_state, before, bits, self.init_bits,
                                 self.crng_state, action))

    # ---- early init (CPU/arch + identity) --------------------------------
    def random_init_early_model(self, command_line, uts_name, arch_rng_words):
        """Models random_init()/add_device_randomness() of cmdline+UTS and the
        CPU/arch RNG seeding. Credit for arch bits is gated by TRUST_CPU."""
        def early(idx, source, data, trust_flag, credit, hinf, fc, note):
            ev = Event(idx=idx, source=source, field_class=fc, adversary_known=(fc!="opaque"),
                       assumed_hinf_given_VA=hinf, comment=note)
            ev.encoded_bits = len(data) * 8
            self.input_pool.mix_pool_bytes(data, source, credited=(credit > 0))
            ev.kernel_credit = credit
            if credit > 0 and self.crng_state != "CRNG_READY":
                self.credit_init_bits_model(credit, idx, "early init credit")
            self.acct.add(ev, note)
            self.early_init_rows.append((source, len(data), trust_flag, credit, hinf, note))
            return ev

        e0 = early(0, "CMDLINE", command_line, "-", 0, 0, "public",
                   "kernel command line: public, mixed, uncredited")
        e1 = early(1, "UTS", uts_name, "-", 0, 0, "public",
                   "UTS/system identity: public, mixed, uncredited")
        if TRUST_CPU:
            e2 = early(2, "ARCH_EARLY", arch_rng_words, "trust_cpu=on",
                       len(arch_rng_words)*8, len(arch_rng_words)*8, "opaque",
                       "arch/CPU RNG: trust_cpu=on -> mixed AND credited")
        else:
            e2 = early(2, "ARCH_EARLY", arch_rng_words, "trust_cpu=off", 0, 0, "predictable",
                       "arch/CPU RNG: trust_cpu=off -> mixed but uncredited")
        return [e0, e1, e2]

    # ---- source taxonomy -------------------------------------------------
    def add_device_randomness_model(self, ev, data):
        ev.encoded_bits = len(data) * 8
        self.input_pool.mix_pool_bytes(data, ev.source, credited=False)
        self.acct.add(ev, "device/identity bytes; adversary-knowable; 0 credit")

    def add_bootloader_randomness_model(self, ev, data):
        # add_bootloader_randomness() -> add_device_randomness(); crediting gated by trust_bootloader.
        ev.encoded_bits = len(data) * 8
        self.input_pool.mix_pool_bytes(data, ev.source, credited=TRUST_BOOTLOADER)
        if TRUST_BOOTLOADER:
            ev.kernel_credit = len(data) * 8
            if self.crng_state != "CRNG_READY":
                self.credit_init_bits_model(ev.kernel_credit, ev.idx, "bootloader credit")
            self.log("   random.trust_bootloader=True  -> mixed and credited (%d bits)" % ev.kernel_credit)
        else:
            self.log("   random.trust_bootloader=False -> mixed but uncredited")
        self.acct.add(ev, "bootloader seed; field_class=%s; credit gated by trust_bootloader" % ev.field_class)

    def add_hwgenerator_randomness_model(self, ev, data, claimed_bits, sleep_after=False):
        ev.encoded_bits = len(data) * 8
        self.input_pool.mix_pool_bytes(data, ev.source, credited=TRUST_HWRNG)
        if TRUST_HWRNG:
            ev.kernel_credit = claimed_bits
            if self.crng_state != "CRNG_READY":
                self.credit_init_bits_model(claimed_bits, ev.idx, "hwrng credit")
            note = "hwrng: TRUST_HWRNG=True -> credits %d bits" % claimed_bits
        else:
            note = "hwrng: TRUST_HWRNG=False -> mixed but 0 credit"
        if sleep_after:
            self.log("   hwrng sleep_after would throttle feeder in real kernel (not actually sleeping)")
        self.acct.add(ev, note)

    def add_vmfork_randomness_model(self, ev, data):
        # VMGENID is uniqueness, not secrecy: mix, never credit; reseed only if already ready.
        ev.encoded_bits = len(data) * 8
        self.input_pool.mix_pool_bytes(data, ev.source, credited=False)
        if self.crng_state == "CRNG_READY":
            self.reseed_seed = self.input_pool.seed(); self.reseed_idx = ev.idx
            self.crng_reseed_model(trigger="vmfork/vmgenid")
            note = "vmgenid: mixed, uncredited, reseeded (was READY)"
        else:
            self.vmfork_seen_pre_ready = True
            note = "vmgenid: mixed, uncredited, pre-ready -> no reseed yet"
        self.log("   VMGENID is uniqueness, not secrecy: mixed, uncredited, reseeds only if ready")
        self.acct.add(ev, note)

    def add_interrupt_randomness_model(self, ev):
        cpu = ev.cpu % NCPU; fp = self.fastpools[cpu]
        m0 = ev.cycles & MASK64
        m1 = (ev.payload["ip"] ^ swab64(ev.payload["irq"])) & MASK64
        fp.mix(m0, m1)
        sim_count = ev.payload.get("sim_count")
        eff_count = sim_count if sim_count is not None else fp.count
        time_elapsed = (ev.jiffies - fp.last_flush_jiffies) if ev.jiffies is not None else 0
        trig_count = eff_count >= FLUSH_COUNT
        trig_time = time_elapsed >= HZ
        flush = ev.payload.get("flush", False) or trig_count or trig_time
        inflight_before = fp.mix_inflight
        executed = False; raw_credit = 0; clamped = 0
        if flush:
            fp.mix_inflight = True                     # scheduled
            raw_credit, clamped = self.mix_interrupt_randomness_model(ev, cpu, sim_count)
            fp.mix_inflight = False                    # worker ran
            executed = True
        self.fastpool_rows.append((ev.idx, cpu, ev.payload["irq"], m0, m1, fp.pool[:],
                                   fp.count, flush, clamped))
        self.flush_rows.append((ev.idx, cpu, eff_count, time_elapsed, trig_count, trig_time,
                                inflight_before, executed, raw_credit, clamped,
                                self.init_bits, self.input_pool.preview()))
        ev.kernel_credit = clamped
        self.acct.add(ev, "interrupt timing; jitter low bits only under V_A")

    def mix_interrupt_randomness_model(self, ev, cpu, sim_count=None):
        fp = self.fastpools[cpu]
        count = sim_count if sim_count is not None else fp.count
        p0, p1 = fp.read_two_longs()
        ev.encoded_bits = FAST_POOL_WORDS_MIXED * 64
        if MIX_COUNT_INTO_POOL:
            self.input_pool.mix_pool_bytes(struct.pack('<I', count & MASK32), ev.source, credited=False)
            ev.encoded_bits += 32
        self.input_pool.mix_pool_bytes(struct.pack('<QQ', p0, p1), ev.source, credited=True)
        raw_credit = max(1, (count & 0xffff) // 64)
        # NOTE: mainline uses just max(1, count/64) (later clamped to POOL_BITS in
        # credit_init_bits). The per-flush clamp below is a conservative modeling
        # choice enabled by CLAMP_IRQ_CREDIT, not mainline behavior.
        clamped = min(raw_credit, FAST_POOL_WORDS_MIXED * 64) if CLAMP_IRQ_CREDIT else raw_credit
        if self.crng_state != "CRNG_READY":
            self.credit_init_bits_model(clamped, ev.idx, "irq flush")
        fp.count = 0
        if ev.jiffies is not None: fp.last_flush_jiffies = ev.jiffies
        return raw_credit, clamped

    def add_timer_randomness_model(self, ev, state_key, num, cycles, jiffies, context):
        """HID/disk path. process -> mix into input_pool + estimate + credit.
        hardirq -> data goes to the per-CPU fast pool; crediting still happens
        (mainline credits init_bits directly in BOTH contexts)."""
        d1 = d2 = d3 = mn = bits = 0
        if context == "hardirq":
            fp = self.fastpools[ev.cpu % NCPU]
            fp.mix(cycles & MASK64, num & MASK64)      # add_timer_randomness in_hardirq -> fast_mix
            ev.encoded_bits = 0
            if self.crng_state != "CRNG_READY":
                st = self.timer_states.setdefault(state_key, [0, 0, 0])
                d1 = jiffies - st[0]; st[0] = jiffies
                d2 = d1 - st[1]; st[1] = d1
                d3 = d2 - st[2]; st[2] = d2
                mn = min(abs(d1), abs(d2), abs(d3)); bits = min(fls(mn >> 1), 11)
                if HARDIRQ_COUNT_INFLATION:
                    extra = max(1, bits * 64) - 1      # spec variant: inflate count, credit via later flush
                    fp.count += extra
                    self.hardirq_rows.append((ev.idx, ev.cpu % NCPU, bits, extra, "count-inflation"))
                    self.log("   hardirq timer event -> fast_mix + count inflation by max(1,bits*64)-1 = %d" % extra)
                    branch = "hardirq->fast_mix+inflate(+%d)" % extra
                else:
                    ev.kernel_credit = bits
                    self.credit_init_bits_model(bits, ev.idx, "hardirq timer credit")
                    self.hardirq_rows.append((ev.idx, ev.cpu % NCPU, bits, 0, "direct-credit"))
                    self.log("   hardirq timer event -> fast_mix(entropy,num); credit_init_bits(%d) directly (mainline)" % bits)
                    branch = "hardirq->fast_mix+credit(%d)" % bits
            else:
                branch = "hardirq->fast_mix (crng ready: no credit)"
        else:
            self.input_pool.mix_pool_bytes(struct.pack('<Q', cycles & MASK64), ev.source, credited=True)
            self.input_pool.mix_pool_bytes(struct.pack('<I', num & MASK32), ev.source, credited=True)
            ev.encoded_bits = 8*8 + 4*8
            branch = "process->mix+estimate"
            if self.crng_state != "CRNG_READY":
                st = self.timer_states.setdefault(state_key, [0, 0, 0])
                d1 = jiffies - st[0]; st[0] = jiffies
                d2 = d1 - st[1]; st[1] = d1
                d3 = d2 - st[2]; st[2] = d2
                mn = min(abs(d1), abs(d2), abs(d3)); bits = min(fls(mn >> 1), 11)
                ev.kernel_credit = bits
                self.credit_init_bits_model(bits, ev.idx, "timer estimate")
        self.timer_rows.append((ev.idx, ev.source, state_key, num, d1, d2, d3, mn,
                                ev.kernel_credit, self.init_bits, branch, ev.ignored_repeat))
        return branch

    def add_input_randomness_model(self, ev):
        p = ev.payload
        value = p["value"]
        # auto-repeat filter: add_input_randomness drops events whose value repeats.
        if value == self.input_last_value:
            ev.ignored_repeat = True; ev.encoded_bits = 0; ev.kernel_credit = 0
            num = ((p["type"] << 4) ^ p["code"] ^ (p["code"] >> 4) ^ value) & MASK32
            self.timer_rows.append((ev.idx, ev.source, "INPUT", num, 0, 0, 0, 0, 0,
                                    self.init_bits, "ignored repeat", True))
            self.acct.add(ev, "auto-repeat (value==last): NOT mixed, NOT credited")
            return
        self.input_last_value = value
        num = ((p["type"] << 4) ^ p["code"] ^ (p["code"] >> 4) ^ value) & MASK32
        self.add_timer_randomness_model(ev, "INPUT", num, ev.cycles, ev.jiffies, ev.context)
        self.acct.add(ev, "HID timing; key id public, jitter is the entropy")

    def add_disk_randomness_model(self, ev):
        devt = ev.payload["devt"]
        if not self._disk_warned:
            self.log("   [fidelity] disk_event_num() = 0x100+devt is a SIMPLIFIED stand-in; pin the exact")
            self.log("              major/minor encoding to your random.c before relying on disk 'num' bytes.")
            self._disk_warned = True
        num = disk_event_num(devt)
        self.add_timer_randomness_model(ev, "DISK:0x%x" % devt, num, ev.cycles, ev.jiffies, ev.context)
        self.acct.add(ev, "disk completion timing; per-devt estimator state")

    def random_write_iter_model(self, ev, data):
        ev.encoded_bits = len(data) * 8
        self.input_pool.mix_pool_bytes(data, ev.source, credited=False)
        self.acct.add(ev, "user write: mixes attacker-chosen bytes, 0 credit (by design)")

    def random_ioctl_model(self, ev):
        p = ev.payload
        if ev.source in ("ADMIN_CREDIT", "ADMIN_ADD_ENTROPY"):
            if not CAP_SYS_ADMIN:
                ev.kernel_credit = 0
                self.denied_ioctls.append((ev.idx, ev.source, "EPERM: no CAP_SYS_ADMIN"))
                self.log("   ioctl %s DENIED: CAP_SYS_ADMIN=False -> EPERM, 0 credit" % ev.source)
                self.acct.add(ev, "ADMIN ioctl denied (no CAP_SYS_ADMIN)")
                return
        if ev.source == "ADMIN_CREDIT":                 # RNDADDTOENTCNT: credit, NO data
            bits = p["add_bits"]; ev.kernel_credit = bits
            if self.crng_state != "CRNG_READY":
                self.credit_init_bits_model(bits, ev.idx, "RNDADDTOENTCNT")
            self.acct.add(ev, "ADMIN RNDADDTOENTCNT: credits bits with NO new data -> trust boundary")
        elif ev.source == "ADMIN_ADD_ENTROPY":          # RNDADDENTROPY: data + credit
            data = bytes(p["data"]); ev.encoded_bits = len(data) * 8
            self.input_pool.mix_pool_bytes(data, ev.source, credited=True)
            bits = p["add_bits"]; ev.kernel_credit = bits
            if self.crng_state != "CRNG_READY":
                self.credit_init_bits_model(bits, ev.idx, "RNDADDENTROPY")
            self.acct.add(ev, "ADMIN RNDADDENTROPY: mixes bytes AND credits -> trust boundary")
        elif ev.source == "RNDRESEEDCRNG":
            if self.crng_state == "CRNG_READY":
                self.reseed_seed = self.input_pool.seed(); self.reseed_idx = ev.idx
                self.crng_reseed_model(trigger="RNDRESEEDCRNG ioctl")
            self.acct.add(ev, "RNDRESEEDCRNG: forces a reseed if already ready")

    # ---- extraction + CRNG ----------------------------------------------
    def extract_entropy_model(self, input_pool, requested_len, arch_rng_block, label):
        seed = input_pool.seed()
        next_key = blake2s(arch_rng_block + struct.pack('<Q', 0), key=seed)
        input_pool.rekey(next_key)
        out = bytearray(); counter = 0; blocks = []
        while len(out) < requested_len:
            counter += 1
            blk = blake2s(arch_rng_block + struct.pack('<Q', counter), key=seed)
            take = min(32, requested_len - len(out)); out += blk[:take]
            blocks.append((counter, blk.hex()))
        if self.verbose:
            self.log(rule()); self.log("extract_entropy(label=%s, len=%d)" % (label, requested_len)); self.log(rule())
            self.log("  seed      = BLAKE2s_final(input_pool)        = %s" % seed.hex())
            self.log("  arch_rng  = (RDSEED/RDRAND/TSC stand-in)     = %s" % arch_rng_block.hex())
            self.log("  next_key  = BLAKE2s(key=seed, arch||cnt=0)   = %s  -> rekeys pool" % next_key.hex())
            for c, h in blocks:
                self.log("  out#%d     = BLAKE2s(key=seed, arch||cnt=%d) = %s" % (c, c, h))
        seed = b'\x00'*32; next_key = b'\x00'*32
        return bytes(out[:requested_len])

    def crng_reseed_model(self, trigger=""):
        before_gen = self.base.generation
        key = self.extract_entropy_model(self.input_pool, 32, ARCH_RNG_BLOCK, "crng_reseed")
        self.base.key = key; self.base.generation += 1
        self.base.birth_time = self.reseed_idx; self.base.ready_state = "CRNG_READY"
        if self.verbose:
            self.log("  crng_reseed: trigger=%s  generation %d -> %d" % (trigger, before_gen, self.base.generation))
            self.log("  base_crng.key = %s" % key.hex())

    def crng_make_state_model(self, cpu, out_len):
        pc = self.percpu[cpu % NCPU]; steps = []
        if pc.generation != self.base.generation:
            pc.key, _s, _d = crng_fast_key_erasure(self.base.key, 32)
            pc.generation = self.base.generation
            steps.append(("derive percpu from base (erase #1)", pc.key.hex()))
        new_key, state, out = crng_fast_key_erasure(pc.key, out_len)
        steps.append(("erase percpu -> output (erase #2)", new_key.hex()))
        pc.key = new_key
        return state, out, steps

    def crng_make_state_pre_ready_model(self, cpu, out_len):
        """Best-effort output BEFORE CRNG_READY. Extracts current (insufficient)
        pool into base_crng.key, does fast key erasure, but does NOT set READY."""
        assert self.crng_state != "CRNG_READY"
        key = self.extract_entropy_model(self.input_pool, 32, ARCH_RNG_BLOCK, "pre_ready_extract")
        self.base.key = key                                   # best-effort key; generation NOT marked ready
        new_key, state, out = crng_fast_key_erasure(self.base.key, min(32, out_len))
        self.base.key = new_key
        self.pre_ready_events.append((cpu, out_len, "pre_ready_best_effort"))
        self.log("  [WARN] pre-ready best-effort output: pool has < 256 credited bits; "
                 "marked INSECURE (GRND_INSECURE / early-boot only)")
        return bytes(out[:out_len]), "pre_ready_best_effort"

    def get_random_bytes_model(self, n, cpu):
        if self.crng_state != "CRNG_READY":
            out, tag = self.crng_make_state_pre_ready_model(cpu, min(32, n))
            return out, [("pre-ready best-effort (kernel caller, warns)", tag)]
        first_len = min(32, n)
        state, buf, steps = self.crng_make_state_model(cpu, first_len)
        buf = bytearray(buf); remaining = n - len(buf)
        while remaining > 0:
            blk = chacha20_block(state)
            if state[12] == 0: state[13] = (state[13] + 1) & MASK32
            take = min(remaining, 64); buf += blk[:take]; remaining -= take
        return bytes(buf[:n]), steps

    def getrandom_model(self, n, mode):
        """Conceptual semantics only. Never touches the OS."""
        ready = (self.crng_state == "CRNG_READY")
        if mode == "blocking":
            return dict(mode=mode, would_block=(not ready), warns=False, returns=ready,
                        insecure=False, errno=None,
                        uses="CRNG_READY pool" if ready else "blocks in wait_for_random_bytes (no return)")
        if mode == "nonblocking":
            return dict(mode=mode, would_block=False, warns=False, returns=ready,
                        insecure=False, errno=(None if ready else "EAGAIN"),
                        uses="CRNG_READY pool" if ready else "returns EAGAIN")
        if mode == "GRND_INSECURE":
            return dict(mode=mode, would_block=False, warns=(not ready), returns=True,
                        insecure=(not ready), errno=None,
                        uses="CRNG_READY pool" if ready else "pre-ready best-effort (INSECURE)")
        if mode == "urandom":
            return dict(mode=mode, would_block=False, warns=(not ready), returns=True,
                        insecure=(not ready), errno=None,
                        uses="CRNG_READY pool" if ready else "pre-ready best-effort (warns once)")
        if mode == "get_random_bytes_kernel":
            return dict(mode=mode, would_block=False, warns=(not ready), returns=True,
                        insecure=(not ready), errno=None,
                        uses="CRNG_READY pool" if ready else "pre-ready best-effort (warns)")
        return dict(mode=mode, would_block=False, warns=False, returns=ready, insecure=False, errno=None, uses="?")


# ===========================================================================
# 6.  HARDCODED EVENT STREAM  (the only fixed part)
# ===========================================================================
def build_events(start_idx):
    E = []; idx = [start_idx]; jpos = [0]
    def add(**kw):
        ev = Event(idx=idx[0], jpos=jpos[0], **kw); E.append(ev)
        idx[0] += 1; jpos[0] += 1; return ev

    add(source="BOOTLOADER", field_class="public", adversary_known=True,
        payload={"data": list(b"BOOT-CMDLINE-seed-v1")},
        comment="bootloader seed; credit gated by trust_bootloader")
    add(source="DEVICE", field_class="public", adversary_known=True,
        payload={"data": list(bytes.fromhex("525400abcdef")) + list(b"eth0-mac")},
        comment="add_device_randomness: MACs/device ids, 0 credit")
    add(source="VMFORK", field_class="predictable", adversary_known=True,
        payload={"data": list(bytes.fromhex("0102030405060708090a0b0c0d0e0f10"))},
        comment="vmgenid pre-ready (uniqueness, not secrecy)")

    add(source="IRQ", cpu=0, jiffies=10, field_class="predictable", adversary_known=False,
        assumed_hinf_given_VA=2, payload={"irq":16,"ip":0xffffffff81234560}, comment="IRQ cpu0")
    add(source="IRQ", cpu=1, jiffies=11, field_class="predictable", adversary_known=False,
        assumed_hinf_given_VA=2, payload={"irq":17,"ip":0xffffffff81234588}, comment="IRQ cpu1")
    add(source="IRQ", cpu=0, jiffies=1100, field_class="predictable", adversary_known=False,
        assumed_hinf_given_VA=2, payload={"irq":18,"ip":0xffffffff812345b0},
        comment="IRQ cpu0 (time-triggered flush: 1100-0 >= HZ)")

    # one HARDIRQ timer event to exercise the hardirq branch (data->fast pool, credit direct)
    add(source="INPUT", cpu=1, jiffies=15000, context="hardirq", field_class="predictable",
        adversary_known=False, assumed_hinf_given_VA=3, payload={"type":1,"code":40,"value":1},
        comment="HID in hardirq context")

    hid = [(5000,1,30,1),(20000,1,31,1),(25000,1,32,0),(41000,1,33,1),
           (47000,1,34,1),(60000,1,35,0),(65000,1,36,1),(81000,1,37,1),
           (88000,1,30,0),(102000,1,31,1),(110000,1,32,1)]
    dsk = [6000,23000,29000,46000,52000,69000,75000,93000,100000,118000]
    hi = [0]; di = [0]
    order = ["H","D","H","D","H","D","H","D","IRQ","H","D","HW","H","D","H","D",
             "H","D","H","D","H","D"]
    for tok in order:
        if tok == "H" and hi[0] < len(hid):
            j,t,c,v = hid[hi[0]]; hi[0] += 1
            add(source="INPUT", jiffies=j, context="process", field_class="predictable",
                adversary_known=False, assumed_hinf_given_VA=3,
                payload={"type":t,"code":c,"value":v}, comment="HID key event")
        elif tok == "D" and di[0] < len(dsk):
            j = dsk[di[0]]; di[0] += 1
            add(source="DISK", jiffies=j, context="process", field_class="predictable",
                adversary_known=False, assumed_hinf_given_VA=3,
                payload={"devt":0x800100}, comment="disk completion")
        elif tok == "IRQ":
            add(source="IRQ", cpu=1, jiffies=2000, field_class="predictable",
                adversary_known=False, assumed_hinf_given_VA=4,
                payload={"irq":20,"ip":0xffffffff81234600,"sim_count":1024},
                comment="IRQ cpu1 big flush (count=1024)")
        elif tok == "HW":
            add(source="HWRNG", field_class="opaque", adversary_known=False,
                assumed_hinf_given_VA=64,
                payload={"data": list(bytes.fromhex("11223344556677889900aabbccddeeff")),
                         "claimed_bits":64,"sleep_after":True},
                comment="hwrng/virtio-rng feeder")

    add(source="USER_WRITE", field_class="public", adversary_known=True,
        payload={"data": list(b"ATTACKER-CONTROLLED-BYTES-000000")},
        comment="write(/dev/urandom): mixes chosen bytes, 0 credit")
    add(source="ADMIN_CREDIT", field_class="public", adversary_known=True,
        assumed_hinf_given_VA=0, payload={"add_bits":128},
        comment="RNDADDTOENTCNT: claims +128 bits with NO data (dishonest under V_A)")
    add(source="VMFORK", field_class="predictable", adversary_known=True,
        payload={"data": list(bytes.fromhex("aabbccddeeff00112233445566778899"))},
        comment="vmgenid AFTER ready -> triggers reseed")
    return E


# ===========================================================================
# 7.  SELF-TESTS
# ===========================================================================
def self_tests():
    print(rule()); print("STEP 0  -  known-answer self-tests (fidelity proof)"); print(rule())
    d = blake2s(b"abc")
    w = bytes.fromhex("508c5e8c327c14e2e1a72ba34eeb452f37458b209ed63a294d999b4c86675982")
    ok1 = (d == w); print(" [%s] BLAKE2s-256(\"abc\")" % ("PASS" if ok1 else "FAIL"))
    k = bytes(range(32)); msg = b"high-fidelity educational model"
    ok2 = (blake2s(msg, key=k) == hashlib.blake2s(msg, key=k, digest_size=32).digest())
    print(" [%s] keyed BLAKE2s vs hashlib.blake2s" % ("PASS" if ok2 else "FAIL"))
    st = chacha_init_state(bytes(range(32)))
    st[12] = 1; st[13] = struct.unpack('<I', bytes.fromhex("00000009"))[0]
    st[14] = struct.unpack('<I', bytes.fromhex("0000004a"))[0]; st[15] = 0
    blk = chacha20_block(st)
    w2 = bytes.fromhex("10f1e7e4d13b5915500fdd1fa32071c4c7d1f4c733c0680304"
                       "22aa9ac3d46c4ed2826446079faa0914c2d705d98b02a2"
                       "b5129cd1de164eb9cbd083e8a2503c4e")
    ok3 = (blk == w2); print(" [%s] ChaCha20 block (RFC 8439 2.3.2)" % ("PASS" if ok3 else "FAIL"))
    assert ok1 and ok2 and ok3, "self-tests failed"
    print()


# ===========================================================================
# 8.  FIDELITY MATRIX + WARNINGS
# ===========================================================================
FIDELITY_MATRIX = [
 ("fast_mix / SipHash perm", "algorithm-exact", "single fixed permutation count", "interrupt mixing shape"),
 ("BLAKE2s-256",            "byte-exact vs hashlib", "none", "input_pool conditioning"),
 ("ChaCha20 block",         "byte-exact vs RFC8439", "none", "output generation"),
 ("fast key erasure",       "structurally faithful", "single-shot per call", "forward secrecy"),
 ("timer estimator",        "formula-faithful", "per-source state simplified", "entropy crediting"),
 ("hardirq timer branch",   "mainline (direct credit)", "inflation variant behind flag", "credit placement"),
 ("extract_entropy",        "structurally faithful", "arch_rng hardcoded; size_t LE", "HKDF-like extract"),
 ("per-CPU fast pools",     "modeled (NCPU=%d)" % NCPU, "no real SMP/locking/workqueue", "flush behavior"),
 ("readiness + pre-ready",  "modeled", "no real scheduler/blocking", "getrandom semantics"),
 ("trust flags",            "modeled (cpu/boot/hwrng)", "boot params not parsed from cmdline", "crediting gates"),
 ("flush trigger",          "modeled (>=1024 or >=1s)", "sim_count hardcoded for big flush", "credit path"),
 ("irq credit clamp",       "modeling choice (flag)", "mainline has NO per-flush clamp", "credit magnitude"),
 ("disk num encoding",      "0x100+devt stand-in", "exact major/minor not pinned", "exact pool bytes"),
 ("arch RNG / RDSEED",      "hardcoded stand-in", "no real RDSEED/RDRAND", "extraction determinism"),
 ("adversary accounting",   "illustrative only", "assumed Hinf is an input", "honesty/ceiling"),
]
def print_target_and_matrix():
    print(rule()); print("TARGET KERNEL"); print(rule())
    for k, v in TARGET_KERNEL.items():
        print("  %-18s : %s" % (k, v))
    if TARGET_KERNEL["validated_against"] is None:
        print("  [WARNING] validated_against is None -> outputs are NOT version-faithful.")
        print("            %s" % TARGET_KERNEL["required_action"])
    print("  trust flags        : TRUST_CPU=%s  TRUST_BOOTLOADER=%s  TRUST_HWRNG=%s  CAP_SYS_ADMIN=%s"
          % (TRUST_CPU, TRUST_BOOTLOADER, TRUST_HWRNG, CAP_SYS_ADMIN))
    print()
    print(rule()); print("FIDELITY MATRIX"); print(rule())
    print(" %-26s | %-24s | %-31s | %s" % ("component","implemented fidelity","simplification","why it matters"))
    print(" " + "-"*26 + "-+-" + "-"*24 + "-+-" + "-"*31 + "-+-" + "-"*20)
    for c,f,s,w in FIDELITY_MATRIX:
        print(" %-26s | %-24s | %-31s | %s" % (c,f,s,w))
    print()


# ===========================================================================
# 9.  REMAINING NON-BIT-EXACT AREAS
# ===========================================================================
UNFAITHFUL_REMAINING = [
    "exact kernel commit not pinned unless TARGET_KERNEL['validated_against'] is set",
    "no real SMP locking",
    "no real workqueue/timer scheduling",
    "no exact distro random.c byte-for-byte validation",
    "no real architecture RNG behavior (RDSEED/RDRAND hardcoded)",
    "no vDSO getrandom",
    "no signal/buffer/error handling for syscalls",
    "no real kernel blocking or scheduler semantics",
    "no exact hardware RNG feeder thread behavior",
    "no exact ioctls beyond conceptual accounting",
    "per-flush IRQ credit clamp is a modeling choice, not mainline",
    "disk 'num' encoding (0x100+devt) is a simplified stand-in",
]
def print_unfaithful_remaining():
    print(rule()); print("REMAINING NON-BIT-EXACT AREAS"); print(rule())
    for item in UNFAITHFUL_REMAINING:
        print("   - %s" % item)
    print()


# ===========================================================================
# 10.  DRIVER
# ===========================================================================
def run_model(jitter=None, verbose=True):
    jitter = jitter if jitter is not None else JITTER
    m = LinuxRNGEducationalModel(jitter, verbose=verbose)
    m.random_init_early_model(KERNEL_COMMAND_LINE, UTS_NAME, ARCH_EARLY_RANDOM)

    events = build_events(start_idx=NUM_EARLY)
    for ev in events:
        ev.cycles = cycles_of(ev.jpos, jitter)

    for ev in events:
        s = ev.source
        if s == "BOOTLOADER":   m.add_bootloader_randomness_model(ev, bytes(ev.payload["data"]))
        elif s == "DEVICE":     m.add_device_randomness_model(ev, bytes(ev.payload["data"]))
        elif s == "VMFORK":     m.add_vmfork_randomness_model(ev, bytes(ev.payload["data"]))
        elif s == "HWRNG":      m.add_hwgenerator_randomness_model(ev, bytes(ev.payload["data"]),
                                                                   ev.payload["claimed_bits"],
                                                                   ev.payload.get("sleep_after", False))
        elif s == "IRQ":        m.add_interrupt_randomness_model(ev)
        elif s == "INPUT":      m.add_input_randomness_model(ev)
        elif s == "DISK":       m.add_disk_randomness_model(ev)
        elif s == "USER_WRITE": m.random_write_iter_model(ev, bytes(ev.payload["data"]))
        elif s in ("ADMIN_CREDIT","ADMIN_ADD_ENTROPY","RNDRESEEDCRNG"): m.random_ioctl_model(ev)

    if verbose: _print_all_tables(m, events)

    final_out = b""; out_steps = []
    if m.crng_state == "CRNG_READY":
        final_out, out_steps = m.get_random_bytes_model(80, cpu=0)

    report = _build_report(m, events, final_out)
    if verbose: _print_output_and_reports(m, events, final_out, out_steps, report)
    return m, events, final_out, report


def _describe(ev):
    s = ev.source
    if s == "IRQ":   return "m0=cycles m1=ip^swab(irq) irq=%d" % ev.payload["irq"]
    if s == "INPUT": p=ev.payload; return "cyc+num (t=%d c=%d v=%d)%s" % (p["type"],p["code"],p["value"], " REPEAT" if ev.ignored_repeat else "")
    if s == "DISK":  return "cyc+num=0x100+devt(0x%x)" % ev.payload["devt"]
    if s == "ADMIN_CREDIT": return "+%d bits, NO data" % ev.payload["add_bits"]
    return "%d bytes" % len(ev.payload.get("data",[]))


def _print_all_tables(m, events):
    # EARLY INIT
    print(rule()); print("EARLY INIT MIXING (random_init: cmdline + UTS + arch/CPU RNG)"); print(rule())
    print(" source     | bytes | trust flag    | credited | assumed_Hinf|VA | note")
    print(" -----------+-------+---------------+----------+----------------+-----------------------------------")
    for (src, nb, tf, cr, hinf, note) in m.early_init_rows:
        print(" %-10s | %5d | %-13s | %8d | %14d | %s" % (src, nb, tf, cr, hinf, note))
    print()

    print(rule()); print("STEP 1  -  source taxonomy: what each hardcoded event hands to the RNG"); print(rule())
    print(" idx | source            | cpu | ctx     | bytes mixed / words                     | credited?")
    print(" ----+-------------------+-----+---------+-----------------------------------------+----------")
    for ev in events:
        if ev.ignored_repeat: credited = "IGNORED"
        elif (ev.kernel_credit or 0) > 0: credited = "yes"
        elif ev.source == "ADMIN_CREDIT": credited = "CREDIT-ONLY"
        else: credited = "no"
        print(" %3d | %-17s | %3s | %-7s | %-39s | %s" %
              (ev.idx, ev.source, ev.cpu, ev.context, _describe(ev)[:39], credited))
    print()

    print(rule()); print("STEP 2  -  per-CPU fast_pool via fast_mix() = SipHash-1-0  (NCPU=%d)" % NCPU); print(rule())
    print(" idx | cpu | irq |    m0 (cycles)     |    m1 (ip^swab)    | count | flush? | credit | pool[0..1]")
    print(" ----+-----+-----+--------------------+--------------------+-------+--------+--------+-----------")
    for (idx,cpu,irq,m0,m1,pool,cnt,flush,credit) in m.fastpool_rows:
        print(" %3d | %3d | %3d | %018x | %018x | %5d | %-6s | %+5d  | %016x %016x" %
              (idx,cpu,irq,m0,m1,cnt,str(flush),credit,pool[0],pool[1]))
    print()

    print(rule()); print("STEP 3  -  mix_interrupt_randomness flush triggers + credit (raw vs clamped)"); print(rule())
    print(" MIX_COUNT_INTO_POOL=%s  CLAMP_IRQ_CREDIT=%s (clamp=%d)" %
          (MIX_COUNT_INTO_POOL, CLAMP_IRQ_CREDIT, FAST_POOL_WORDS_MIXED*64))
    print(" idx | cpu | count | t_elapsed | trg_cnt | trg_time | inflight_b4 | executed | raw | clmp | init_bits")
    print(" ----+-----+-------+-----------+---------+----------+-------------+----------+-----+------+----------")
    for (idx,cpu,cnt,te,tc,tt,ib,ex,raw,clmp,initb,pv) in m.flush_rows:
        if not ex: continue
        print(" %3d | %3d | %5d | %9d | %-7s | %-8s | %-11s | %-8s | %3d | %4d | %4d/256" %
              (idx,cpu,cnt,te,str(tc),str(tt),str(ib),str(ex),raw,clmp,initb))
    print()

    print(rule()); print("STEP 4  -  add_timer_randomness: delta/delta2/delta3, bits=min(fls(min>>1),11)"); print(rule())
    print(" idx | src   | state_key   |   num    |  delta  | delta2  | delta3  |  min   | bits | ignored | branch")
    print(" ----+-------+-------------+----------+---------+---------+---------+--------+------+---------+--------------------------")
    for (idx,src,sk,num,d1,d2,d3,mn,bits,ib,branch,ign) in m.timer_rows:
        print(" %3d | %-5s | %-11s | %8d | %7d | %7d | %7d | %6d | %4d | %-7s | %s" %
              (idx,src,sk[:11],num,d1,d2,d3,mn,bits,str(ign),branch))
    print()

    if m.hardirq_rows:
        print(rule()); print("STEP 4a -  hardirq timer events (data->fast pool; crediting as configured)"); print(rule())
        print(" idx | cpu | bits | extra_count | mode")
        print(" ----+-----+------+-------------+----------------")
        for (idx,cpu,bits,extra,mode) in m.hardirq_rows:
            print(" %3d | %3d | %4d | %11d | %s" % (idx,cpu,bits,extra,mode))
        print()

    print(rule()); print("STEP 4b -  input_pool (BLAKE2s) lifecycle: pool8 before/after; credited is SEPARATE"); print(rule())
    print(" source            | nbytes | pool8 before | pool8 after  | credited?")
    print(" ------------------+--------+--------------+--------------+----------")
    for (src,nb,b8,a8,cred) in m.input_pool.log:
        print(" %-17s | %6d | %-12s | %-12s | %s" % (src,nb,b8,a8,"yes" if cred else "no"))
    print()

    print(rule()); print("STEP 5  -  readiness transitions (CRNG_EMPTY -> CRNG_EARLY@128 -> CRNG_READY@256)"); print(rule())
    print(" idx | before state | bits before | event credit | bits after | after state  | action")
    print(" ----+--------------+-------------+--------------+------------+--------------+----------------------")
    for (idx,bs,bb,cr,ba,as_,act) in m.transitions:
        print(" %3d | %-12s | %11d | %+12d | %10d | %-12s | %s" % (idx,bs,bb,cr,ba,as_,act))
    print(" final: init_bits=%d/256  crng_state=%s  base_generation=%d  vmfork_pre_ready=%s" %
          (m.init_bits, m.crng_state, m.base.generation, m.vmfork_seen_pre_ready))
    print()


def _security_label(hinf):
    if hinf < POOL_EARLY_BITS: return "toy insecure (< 128 assumed real bits)"
    if hinf < POOL_READY_BITS: return "below 256-bit target (>=128, <256 assumed real bits)"
    return "not rejected by this toy model (>=256 assumed real bits)"


def _build_report(m, events, final_out):
    acct_rows = [dict(idx=i, source=s, encoded_bits=e, kernel_credit=cr,
                      assumed_Hinf_given_VA=h, field_class=fc, notes=n)
                 for (i,s,e,cr,h,fc,n) in m.acct.rows]
    return {
        "target_kernel": TARGET_KERNEL,
        "trust_flags": {"TRUST_CPU":TRUST_CPU,"TRUST_BOOTLOADER":TRUST_BOOTLOADER,
                        "TRUST_HWRNG":TRUST_HWRNG,"CAP_SYS_ADMIN":CAP_SYS_ADMIN},
        "config": {"NCPU":NCPU,"MIX_COUNT_INTO_POOL":MIX_COUNT_INTO_POOL,
                   "HARDIRQ_COUNT_INFLATION":HARDIRQ_COUNT_INFLATION,"CLAMP_IRQ_CREDIT":CLAMP_IRQ_CREDIT,
                   "FAST_POOL_WORDS_MIXED":FAST_POOL_WORDS_MIXED,
                   "POOL_EARLY_BITS":POOL_EARLY_BITS,"POOL_READY_BITS":POOL_READY_BITS,"HZ":HZ},
        "early_init_rows": [dict(source=s,bytes=nb,trust_flag=tf,credited=cr,assumed_Hinf=h,note=n)
                            for (s,nb,tf,cr,h,n) in m.early_init_rows],
        "input_events": [ev.to_dict() for ev in events],
        "ignored_input_repeats": [ev.idx for ev in events if ev.ignored_repeat],
        "interrupt_flush_triggers": [dict(idx=i,cpu=c,count=cnt,time_elapsed=te,trigger_count=tc,
                                          trigger_time=tt,inflight_before=ib,flush_executed=ex,
                                          raw_credit=raw,clamped_credit=clmp,init_bits=initb)
                                     for (i,c,cnt,te,tc,tt,ib,ex,raw,clmp,initb,pv) in m.flush_rows],
        "hardirq_timer_inflation_rows": [dict(idx=i,cpu=c,bits=b,extra_count=x,mode=md)
                                         for (i,c,b,x,md) in m.hardirq_rows],
        "pre_ready_output_events": [dict(cpu=c,out_len=l,tag=t) for (c,l,t) in m.pre_ready_events],
        "denied_privileged_ioctls": [dict(idx=i,source=s,reason=r) for (i,s,r) in m.denied_ioctls],
        "per_event_accounting": acct_rows,
        "state_transitions": [dict(idx=i,before=b,bits_before=bb,credit=cr,bits_after=ba,after=a,action=act)
                              for (i,b,bb,cr,ba,a,act) in m.transitions],
        "final_state": {"crng_state":m.crng_state,"init_bits":m.init_bits,
                        "base_generation":m.base.generation,"reseed_idx":m.reseed_idx,
                        "vmfork_seen_pre_ready":m.vmfork_seen_pre_ready},
        "final_output_hex": final_out.hex(),
        "honesty_report": {"cumulative_encoded_bits": m.acct.cum_encoded,
                           "cumulative_credited_bits": m.acct.cum_credit,
                           "cumulative_assumed_Hinf_given_VA": m.acct.cum_hinf,
                           "honesty_gap_credit_minus_Hinf": m.acct.honesty_gap,
                           "honest_under_assumptions": m.acct.honesty_gap <= 0},
        "security_report": {"assumed_real_entropy_bits": m.acct.cum_hinf,
                            "label": _security_label(m.acct.cum_hinf)},
        "fidelity_matrix": [dict(component=c,fidelity=f,simplification=s,why=w) for c,f,s,w in FIDELITY_MATRIX],
        "unfaithful_remaining": UNFAITHFUL_REMAINING,
    }


def _print_output_and_reports(m, events, final_out, out_steps, report):
    # pre-ready semantics: demonstrate on a fresh model kept at CRNG_EMPTY
    print(rule()); print("PRE-READY OUTPUT SEMANTICS (conceptual; demonstrated on an EMPTY sub-model)"); print(rule())
    pre = LinuxRNGEducationalModel(JITTER, verbose=False)
    pre.random_init_early_model(KERNEL_COMMAND_LINE, UTS_NAME, ARCH_EARLY_RANDOM)
    for ev in build_events(NUM_EARLY)[:2]:               # process a little, stay EMPTY
        ev.cycles = cycles_of(ev.jpos, JITTER)
        if ev.source == "BOOTLOADER": pre.add_bootloader_randomness_model(ev, bytes(ev.payload["data"]))
        elif ev.source == "DEVICE":   pre.add_device_randomness_model(ev, bytes(ev.payload["data"]))
    print(" sub-model crng_state = %s (init_bits=%d)" % (pre.crng_state, pre.init_bits))
    print(" call                            | would_block | warns | returns | insecure | errno  | uses")
    print(" --------------------------------+-------------+-------+---------+----------+--------+----------------------------")
    for mode in ["blocking","nonblocking","GRND_INSECURE","urandom","get_random_bytes_kernel"]:
        r = pre.getrandom_model(32, mode)
        print(" getrandom/%-21s | %-11s | %-5s | %-7s | %-8s | %-6s | %s" %
              (mode, str(r["would_block"]), str(r["warns"]), str(r["returns"]),
               str(r["insecure"]), str(r["errno"]), r["uses"]))
    pr_out, tag = pre.crng_make_state_pre_ready_model(0, 16)
    print(" GRND_INSECURE pre-ready sample (INSECURE) = %s  [%s]" % (pr_out.hex(), tag))
    print()

    print(rule()); print("STEP 6  -  CRNG output (post-ready): per-CPU fast key erasure, get_random_bytes(80)"); print(rule())
    if m.crng_state != "CRNG_READY":
        print(" [!] never reached CRNG_READY; no post-ready output generated."); print(); return
    print(" base_crng.key = %s  (generation=%d)" % (m.base.key.hex(), m.base.generation))
    for label, kh in out_steps:
        print("   %-36s -> key %s" % (label, kh))
    print(" get_random_bytes(80) on cpu0:")
    for off in range(0, len(final_out), 16):
        print("   %04x: %s" % (off, final_out[off:off+16].hex()))
    print()

    print(rule()); print("USER INTERFACE SEMANTICS (post-ready; conceptual)"); print(rule())
    print(" call                            | would_block | warns | returns | insecure | uses")
    print(" --------------------------------+-------------+-------+---------+----------+----------------------")
    for mode in ["blocking","nonblocking","GRND_INSECURE","urandom","get_random_bytes_kernel"]:
        r = m.getrandom_model(32, mode)
        print(" getrandom/%-21s | %-11s | %-5s | %-7s | %-8s | %s" %
              (mode, str(r["would_block"]), str(r["warns"]), str(r["returns"]), str(r["insecure"]), r["uses"]))
    print()

    print(rule()); print("PRIVILEGED / ADMIN TRUST BOUNDARIES (illustration only; CAP_SYS_ADMIN=%s)" % CAP_SYS_ADMIN); print(rule())
    print("   USER_WRITE        : mixes attacker-chosen bytes, credits 0 (safe by design)")
    print("   ADMIN_CREDIT      : RNDADDTOENTCNT credits bits with NO data -> can inflate the estimate")
    print("   ADMIN_ADD_ENTROPY : RNDADDENTROPY mixes bytes AND credits them")
    print("   RNDRESEEDCRNG     : forces a reseed if already ready")
    if m.denied_ioctls:
        print("   DENIED (no CAP_SYS_ADMIN):")
        for (i,s,r) in m.denied_ioctls: print("     idx %d %s -> %s" % (i,s,r))
    print()

    print(rule()); print("ADVERSARY-VIEW ACCOUNTING (illustrative; NOT Linux internal truth)"); print(rule())
    print(" idx | source            | encoded | kernel_credit | assumed_Hinf|VA | class      | notes")
    print(" ----+-------------------+---------+---------------+----------------+------------+--------------------")
    for (i,s,e,cr,h,fc,n) in m.acct.rows:
        print(" %3d | %-17s | %7d | %13d | %14d | %-10s | %s" % (i,s,e,cr,h,fc,n[:20]))
    print(" " + "-"*108)
    print(" cumulative encoded bits           : %d" % m.acct.cum_encoded)
    print(" cumulative kernel-credited bits   : %d" % m.acct.cum_credit)
    print(" cumulative assumed Hinf(E | V_A)  : %d" % m.acct.cum_hinf)
    print(" honesty gap (credit - assumed)    : %+d  -> %s" %
          (m.acct.honesty_gap, "HONEST" if m.acct.honesty_gap <= 0 else "DISHONEST under these assumptions"))
    print()

    out_bits = len(final_out) * 8
    print(rule()); print("HONESTY / ENTROPY-CEILING / SECURITY REPORT"); print(rule())
    print(" B. Entropy ceiling:")
    print("    assumed real conditional min-entropy under V_A : %d bits" % m.acct.cum_hinf)
    print("    output produced                                : %d bytes (%d apparent bits)" % (len(final_out), out_bits))
    print("    REAL entropy ceiling of the output             : min(input Hinf, 256) = %d bits" % min(m.acct.cum_hinf,256))
    print("    -> ChaCha stretches %d real bits into %d output bits; the ceiling stays %d." %
          (min(m.acct.cum_hinf,256), out_bits, min(m.acct.cum_hinf,256)))
    print(" C. Honesty:")
    print("    credited=%d  assumed Hinf=%d  gap=%+d" % (m.acct.cum_credit, m.acct.cum_hinf, m.acct.honesty_gap))
    print("    verdict: %s" % ("honest under the hardcoded assumptions" if m.acct.honesty_gap <= 0
                                else "DISHONEST under the hardcoded assumptions (credit exceeds assumed real entropy)"))
    print(" D. Security:")
    print("    label: %s" % _security_label(m.acct.cum_hinf))
    print("    NOTE: This illustrates the condition under which a deterministic RNG with input would be")
    print("          dishonest or insecure. It does NOT say Linux is broken.")
    print()


# ===========================================================================
# 11.  EXTRA TESTS
# ===========================================================================
def extra_tests():
    print(rule()); print("STEP 7  -  determinism / perturbation / ceiling tests"); print(rule())
    _, _, out_a, rep0 = run_model(verbose=False)
    _, ev0, out_b, _ = run_model(verbose=False)
    print(" A. determinism replay : two runs identical = %s" % (out_a == out_b))
    reseed_idx = rep0["final_state"]["reseed_idx"]
    flip_ev = next(ev for ev in ev0
                   if ev.source in ("IRQ","INPUT","DISK") and not ev.ignored_repeat
                   and ev.jpos is not None and (reseed_idx is None or ev.idx <= reseed_idx))
    flipped = JITTER[:]; flipped[flip_ev.jpos] ^= 1
    _, _, out_c, _ = run_model(jitter=flipped, verbose=False)
    hd = hamming(out_a, out_c)
    print(" A'. perturbation      : flip 1 jitter bit on event %d (%s) -> differs=%s, Hamming=%d/%d"
          % (flip_ev.idx, flip_ev.source, out_a != out_c, hd, len(out_a)*8))
    _, _, out_d, rep = run_model(verbose=False)
    out_bits = len(out_d)*8; hinf = rep["honesty_report"]["cumulative_assumed_Hinf_given_VA"]
    print(" E. entropy ceiling    : output bits=%d, assumed input Hinf=%d, real ceiling=%d (pseudoentropy=%s)"
          % (out_bits, hinf, min(hinf,256), out_bits > hinf))
    print()


# ===========================================================================
# 12.  GUARDRAILS + JSON + MAIN
# ===========================================================================
def print_guardrails():
    print(rule()); print("BIT-EXACTNESS GUARDRAIL"); print(rule())
    print(" This simulator is not bit-exact to any live Linux kernel unless validated against a")
    print(" pinned random.c version and exact source-path test vectors.")
    print(" To become bit-exact, validate:")
    for item in ["random.c commit hash","exact fast_mix implementation",
        "exact BLAKE2s implementation and init parameters","exact extract_entropy layout and counter size",
        "exact crng_make_state behavior","exact fast key erasure behavior",
        "exact per-CPU CRNG generation logic","exact readiness transitions",
        "exact architecture RNG fallback behavior","exact boot parameters trust_cpu/trust_bootloader",
        "exact ioctls and getrandom behavior"]:
        print("   [ ] %s" % item)
    print()

def write_json(report, path="linux_rng_pipeline_report.json"):
    with open(path, "w") as f: json.dump(report, f, indent=2)
    print(" JSON artifact written: %s" % path)

def main():
    print(); print("#" * 114)
    print("#  high-fidelity educational simulator of selected modern Linux RNG mechanisms")
    print("#  (hardcoded inputs; NOT bit-exact; does not touch the OS RNG)")
    print("#" * 114); print()
    self_tests()
    print_target_and_matrix()
    m, events, final_out, report = run_model(verbose=True)
    extra_tests()
    print_guardrails()
    print_unfaithful_remaining()
    write_json(report)
    print("\n done. same hardcoded inputs -> identical output every run.")

if __name__ == "__main__":
    main()


##################################################################################################################
#  high-fidelity educational simulator of selected modern Linux RNG mechanisms
#  (hardcoded inputs; NOT bit-exact; does not touch the OS RNG)
##################################################################################################################

STEP 0  -  known-answer self-tests (fidelity proof)
 [PASS] BLAKE2s-256("abc")
 [PASS] keyed BLAKE2s vs hashlib.blake2s
 [PASS] ChaCha20 block (RFC 8439 2.3.2)

TARGET KERNEL
  family             : Linux modern random.c post-5.18
  reference          : drivers/char/random.c
  validated_against  : None
  required_action    : Set to exact commit hash or distro source before claiming version fidelity
  [WARNING] validated_against is None -> outputs are NOT version-faithful.
            Set to exact commit hash or distro source before claiming version fidelity
  trust flags        : TRUST_CPU=False  TRUST_BOOTLOADER=False